# 03 · Validate Bronze

**Layer:** Bronze &nbsp;|&nbsp; **Stage:** data quality gate &nbsp;|&nbsp; **Target:** `banking_bronze.dq_results`

The gate between raw landing and any downstream promotion. Every check
produces a row in `dq_results` with an observed value, a threshold, and a
verdict, so a failing run is self-documenting rather than a stack trace.

The final cell emits `DQ_STATUS` — the pipeline's `bronze_quality_gate`
condition task reads that value and branches: on `PASS` the silver builds
start, on `FAIL` the run diverts to `10_quarantine_bad_records`.

**Checks performed**

- both bronze tables are non-empty
- primary keys are populated (`account_id`, `txn_id`)
- duplicate rate on each primary key is within tolerance
- currency codes are on the approved ISO-4217 whitelist
- transaction amounts are strictly positive
- every transaction resolves to a known account


In [ ]:
from pyspark.sql import SparkSession, functions as F
from datetime import datetime

spark = SparkSession.builder.appName("medallion-validate-bronze").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE     = "2026-09-03"
BATCH_ID     = f"batch_{RUN_DATE.replace('-', '')}"
BRONZE_DB    = "banking_bronze"
DQ_THRESHOLD = 0.98
VALID_CURRENCIES = ["INR", "USD", "EUR", "GBP", "AED", "SGD"]

accounts = spark.table(f"{BRONZE_DB}.raw_accounts")
txns     = spark.table(f"{BRONZE_DB}.raw_transactions")

acct_total = accounts.count()
txn_total  = txns.count()
print(f"accounts: {acct_total}    transactions: {txn_total}")

In [ ]:
checks = []

def record(check_name, table, observed, threshold, passed, detail=""):
    """Append one check result. `observed` is the measured pass-ratio or count."""
    checks.append((
        BATCH_ID, RUN_DATE, table, check_name,
        float(observed), float(threshold),
        "PASS" if passed else "FAIL", detail,
    ))
    flag = "PASS" if passed else "FAIL"
    print(f"  [{flag}] {table:<18} {check_name:<28} observed={observed:.4f} threshold={threshold}")

In [ ]:
print("volume checks")

record("table_not_empty", "raw_accounts", acct_total, 1, acct_total > 0,
       f"{acct_total} rows landed")
record("table_not_empty", "raw_transactions", txn_total, 1, txn_total > 0,
       f"{txn_total} rows landed")

In [ ]:
print("completeness checks")

acct_pk_populated = accounts.filter(F.col("account_id").isNotNull()).count()
acct_pk_ratio = acct_pk_populated / acct_total if acct_total else 0.0
record("primary_key_not_null", "raw_accounts", acct_pk_ratio, DQ_THRESHOLD,
       acct_pk_ratio >= DQ_THRESHOLD,
       f"{acct_total - acct_pk_populated} null account_id")

txn_pk_populated = txns.filter(F.col("txn_id").isNotNull()).count()
txn_pk_ratio = txn_pk_populated / txn_total if txn_total else 0.0
record("primary_key_not_null", "raw_transactions", txn_pk_ratio, DQ_THRESHOLD,
       txn_pk_ratio >= DQ_THRESHOLD,
       f"{txn_total - txn_pk_populated} null txn_id")

In [ ]:
print("uniqueness checks")

acct_distinct = accounts.filter(F.col("account_id").isNotNull()).select("account_id").distinct().count()
acct_uniq_ratio = acct_distinct / acct_pk_populated if acct_pk_populated else 0.0
record("primary_key_unique", "raw_accounts", acct_uniq_ratio, DQ_THRESHOLD,
       acct_uniq_ratio >= DQ_THRESHOLD,
       f"{acct_pk_populated - acct_distinct} duplicate account_id")

txn_distinct = txns.filter(F.col("txn_id").isNotNull()).select("txn_id").distinct().count()
txn_uniq_ratio = txn_distinct / txn_pk_populated if txn_pk_populated else 0.0
record("primary_key_unique", "raw_transactions", txn_uniq_ratio, DQ_THRESHOLD,
       txn_uniq_ratio >= DQ_THRESHOLD,
       f"{txn_pk_populated - txn_distinct} duplicate txn_id")

In [ ]:
print("validity checks")

valid_ccy = txns.filter(F.upper(F.col("currency")).isin(VALID_CURRENCIES)).count()
ccy_ratio = valid_ccy / txn_total if txn_total else 0.0
record("currency_in_whitelist", "raw_transactions", ccy_ratio, DQ_THRESHOLD,
       ccy_ratio >= DQ_THRESHOLD,
       f"{txn_total - valid_ccy} rows outside {VALID_CURRENCIES}")

positive_amt = txns.filter(F.col("amount") > 0).count()
amt_ratio = positive_amt / txn_total if txn_total else 0.0
record("amount_is_positive", "raw_transactions", amt_ratio, DQ_THRESHOLD,
       amt_ratio >= DQ_THRESHOLD,
       f"{txn_total - positive_amt} rows with amount <= 0")

In [ ]:
print("referential integrity checks")

known_accounts = accounts.select("account_id").distinct()
resolved = txns.join(known_accounts, on="account_id", how="left_semi").count()
ref_ratio = resolved / txn_total if txn_total else 0.0
record("txn_account_resolves", "raw_transactions", ref_ratio, DQ_THRESHOLD,
       ref_ratio >= DQ_THRESHOLD,
       f"{txn_total - resolved} transactions reference an unknown account")

In [ ]:
dq = spark.createDataFrame(
    checks,
    "batch_id string, run_date string, table_name string, check_name string, "
    "observed double, threshold double, status string, detail string",
).withColumn("checked_at", F.current_timestamp())

(
    dq.write
    .mode("overwrite")
    .format("parquet")
    .saveAsTable(f"{BRONZE_DB}.dq_results")
)

dq.select("table_name", "check_name", "observed", "threshold", "status", "detail") \
  .orderBy("table_name", "check_name").show(50, truncate=False)

In [ ]:
failed = [c for c in checks if c[6] == "FAIL"]
DQ_STATUS = "FAIL" if failed else "PASS"

print("=" * 60)
print(f"checks run    : {len(checks)}")
print(f"checks failed : {len(failed)}")
for c in failed:
    print(f"    - {c[2]}.{c[3]}: {c[7]}")
print(f"DQ_STATUS={DQ_STATUS}")
print("=" * 60)

# The pipeline's condition task reads this value to choose its branch.
dbutils_output = {"dq_status": DQ_STATUS, "failed_checks": len(failed)}
print(dbutils_output)